In [1]:
import logging
import time
from dataclasses import dataclass,field
from abc import ABC, abstractmethod
from typing import Any, Dict, Optional,Union,Literal
import json
from tqdm import tqdm
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datasets import Dataset
# Pushing data to PostGrad SQL
import psycopg
import uuid
import os
from dotenv import load_dotenv
load_dotenv('.env')

import warnings
warnings.filterwarnings("ignore")

/mnt/d/yota/projects/rag_projects/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline,Pipeline

In [3]:
model_id = 'Qwen/Qwen3-4B-Instruct-2507'
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype="bfloat16",
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.3,
    max_new_tokens=1024,
    max_length=None,
)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1224.58it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'max_length', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
from src.suggestor import FinanceAgentInput,FinanceAgentOutput,FinanceAgent

In [ ]:
finance_input = FinanceAgentInput(
    # company_table_name= 'company_table',
    company_lists_csv = "https://archives.nseindia.com/content/indices/ind_nifty50list.csv",
    stock_table_name= 'stock_news_dataset',
)
agent =FinanceAgent(model=pipe)
finance_out = await agent.execute(finance_input)

2026-07-04 17:30:28,869 - INFO - PSQL Connected Successfully!
2026-07-04 17:30:28,943 - INFO - Company Table loaded with 50 data points.
2026-07-04 17:30:28,943 - INFO - Starting Analysis...
Company :'Adani Enterprises Ltd.' News article :   0%|          | 0/52 [00:00<?, ?it/s][transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Company :'Adani Enterprises Ltd.' News article : 100%|██████████| 52/52 [09:44<00:00, 11.24s/it]
Company :'Adani Ports and Special Economic Zone Ltd.' News article : 100%|██████████| 22/22 [04:20<00:00, 11.83s/it]
Company :'Apollo Hospitals Enterprise Ltd.' News article : 100%|██████████| 23/23 [03:46<00:

In [ ]:
from src import BaseAgent, AgentInput, AgentOutput
from src import UploadingInput, UploadToPLSQL
from datetime import date

##### System Prompt

In [30]:
# Find relavalnce
SYSTEM_PROMPT = """You are a Senior Financial Market Analyst.

Your task is to analyze ONE financial news article for ONE company.

Evaluate the article only from an investment perspective.

Ignore emotional language.

Determine whether the news is expected to have a positive, negative, or neutral impact on the company or in the sector.

You have to estimate the confidance based on :
    - Clarity of the article
    - Explicit financial impact
    - Presence of concrete facts
    - Ambiguity

Consider:
    - Earnings
    - Revenue
    - Product Launch
    - Partnership
    - Acquisition
    - Regulation
    - Lawsuit
    - Competition
    - Supply Chain
    - Analyst Ratings
    - CEO Changes
    - Market Expansion
    - Share Buybacks
    - Dividends


You have to return only in JSON format without any extra comment like below : 
{{
    "sentiment": Positive / Negagtive / Neutral,
    "sentiment_score": 0.00,
    "confidence": 0.00,
    "event_type": The Type of event described by the Article,
    "impact": Impact of the Article on the Company. High / Medium / Low / No Impact,
    "time_horizon": "Medium",
    "key_positive_factors": [],
    "key_negative_factors": [],
    "reasoning": A described resoning about your analysis.
}}
"""

SUGGESTER_SYSTEM_PROMPT = """You are a Senior Financial Market Analyst and Stock Market Investor.

Your task is to analyze the positive and negative factors of a stock generated by a Junior Financial Analyst and determine whether the stock is a good investment.

You will be provided with the following information:

- List of Positive Factors
- List of Negative Factors
- List of Reasoning created by the Junior Analyst

Instructions:
- Ignore emotional or sensational language.
- Base your decision only on the provided factors and reasoning.
- Consider both short-term and long-term implications before making a recommendation.

Return your response strictly in the following JSON format:

{{
    "Investment": "Yes" | "No" | "Tentative",
    "Time Period": "1-2 Weeks" | "1-2 Months" | "1-2 Years" | "2+ Years",
    "Expected Profit": "<Estimated percentage increase in stock value>"
}}
"""


##### Prompts

In [32]:
PROMPT = """
Company Details : 
    - Name : {company_name}
    - Industry : {industry}

Article : 
    - Title : {title}
    - Content : {content}
"""

SUGGESTER_PROMPT = """Please analyze the news regarding the company "{company}" from "{industry}" industry.

Positive factors : 
{positive_factors}

Negagtive Factors : 
{negagtive_factors}

Reasoning : 
{reasonings}
"""

### Agegnts

In [ ]:
@dataclass
class AnalystAgentInput(AgentInput):
    # company_table_name:str
    company_df:pd.DataFrame
    stock_table_name:str

@dataclass
class AnalystAgentOutput(AgentOutput):
    market_news:Dict

class AnalystAgent(BaseAgent):
    def __init__(self,model:Pipeline):
        super().__init__("Analyst Agent")
        self.model=model
        
        self.conn = psycopg.connect(
                                        host=os.getenv("PS_HOST"),
                                        port=os.getenv("PS_PORT"),
                                        dbname=os.getenv("PS_DBNAME"),
                                        user=os.getenv("PS_USER"),
                                        password=os.getenv("PS_PASSWORD"),
                                    )
        
        self.company_query = """
                        SELECT *
                        FROM {company_table};
                        """
        self.news_query = """
                    SELECT *
                    FROM {stock_table}
                    WHERE SECTOR LIKE '{industry}%'
                    """
        
        self.market_news = dict()
    
    async def execute(self,input_data:AnalystAgentInput) ->AnalystAgentOutput:
        
        self.logger.info(f'Company Table loaded with {input_data.company_df.shape[0]} data points.')

        self.logger.info('Starting Analysis...')

        for idx,row in  input_data.company_df.iterrows():
            company = row['Company Name']
            industry = row['Industry']
            self.market_news[company] = {
                'company' : company,
                'industry': industry,
                'score' : 0.0,
                'sentiments': [],
            }
            news_df = pd.read_sql_query(
                self.news_query.format(
                    stock_table=input_data.stock_table_name,
                    industry=industry
                    ), self.conn
                )

            for idx,row in tqdm(news_df.iterrows(),total=len(news_df), desc=f"Company :'{company}' News article : "):
                message = [
                    {
                        'role':'system',
                        'content' : SYSTEM_PROMPT,
                    },
                    {
                        'role':'user',
                        'content': PROMPT.format(company_name = company,
                                        industry = industry, 
                                        title = row.title, 
                                        content = row.content),
                    },
                ]

                result = self.model(message)
                sentiment = json.loads(result[0]['generated_text'][-1]['content'])
                if sentiment['sentiment'].lower()!='neutral':
                    self.market_news[company]['sentiments'].append(sentiment)
                
            self.market_news[company]['score'] = 0 if len(self.market_news[company]['sentiments'])==0 else sum([sentiment['sentiment_score'] * sentiment['confidence'] for sentiment in self.market_news[company]['sentiments']]) / len(self.market_news[company]['sentiments'])


        return AnalystAgentOutput(
            market_news = self.market_news,
        )


In [43]:
@dataclass
class SuggesterAgentInput(AgentInput):
    market_news:Dict
@dataclass
class SuggesterAgentOutput(AgentOutput):
    suggestions:Dict

class SuggesterAgent(BaseAgent):
    def __init__(self,model:Pipeline):
        super().__init__("Finance Agent")
        self.model=model
        self.company_suggestions = dict()
    
    async def execute(self,input_data:SuggesterAgentInput) ->SuggesterAgentOutput:
        market_news = input_data.market_news
        for company,resoning in market_news.items():
            industry = resoning['industry']
            positive_factors = [ factor for sentiment in resoning['sentiments'] for factor in sentiment['key_positive_factors']]
            negagtive_factors = [ factor for sentiment in resoning['sentiments'] for factor in sentiment['key_negative_factors']]
            reasonings = [
                sentiment["reasoning"]
                for sentiment in resoning["sentiments"]
                if sentiment["sentiment"].lower() != "neutral"
            ]

            positive_factors = '  - ' +"\n  - ".join(positive_factors)
            negagtive_factors = '  - ' +"\n  - ".join(negagtive_factors)
            reasonings = '  - ' +"\n  - ".join(reasonings)
        
            message  = [
                        {
                            'role':'system',
                            'content':SUGGESTER_SYSTEM_PROMPT,
                        },
                        {
                            'role': 'user',
                            'content': SUGGESTER_PROMPT.format(
                                company=company,
                                industry = industry,
                                positive_factors = positive_factors,
                                negagtive_factors = negagtive_factors,
                                reasonings = reasonings,
                            ),
                        }
                    ]
            result = self.model(message)
            output = json.loads(result[0]['generated_text'][-1]['content'])

            self.company_suggestions[company] = output

        return SuggesterAgentOutput(
            suggestions = self.company_suggestions
            )

In [ ]:
@dataclass
class FinanceAgentInput(AgentInput):
    # company_table_name:str
    company_lists_csv:str
    stock_table_name:str
@dataclass
class FinanceAgentOutput(AgentOutput):
    stocks:Dict

class FinanceAgent(BaseAgent):
    def __init__(self,model:Pipeline):
        super().__init__("Finance Agent")
        self.model=model
        self.analyst_agent = AnalystAgent(self.model)
        self.suggestion_agent = SuggesterAgent(self.model)
        self.psql_uploader = UploadToPLSQL()
    
    async def execute(self,input_data:FinanceAgentInput) ->FinanceAgentOutput:
        company_df = pd.read_csv(input_data.company_lists_csv)
        analyst_input = AnalystAgentInput(
            # company_table_name=input_data.company_table_name,
            company_df = company_df,
            stock_table_name=input_data.stock_table_name,
        )
        analyst_out = await self.analyst_agent.execute(analyst_input)

        suggestion_input = SuggesterAgentInput(
            market_news=analyst_out.market_news
        )

        suggegstion_output = await self.suggestion_agent.execute(suggestion_input)

        investment_sugggestions = pd.DataFrame(
            [[key,value['Investment'],value['Time Period'],value['Expected Profit']] for key,value in suggegstion_output.suggestions.items()],
            columns = ['Industry','Investment','Time Period','Expected Profit']
        )
        investment_sugggestions['Date'] = date.today()

        self.logger.info('Uploading to PSQL...')

        psql_input = UploadingInput(
                                dataset = investment_sugggestions,
                                uploading_columns = ['Industry','Investment',	'Time Period',	'Expected Profit',	'Date'],
                                sql_column_names = ['id', 'company','investment','time_period','expected_profit','created_at'],
                                table_name= 'stock_suggestions',
                            )
        
        self.psql_uploader._truncate_table(psql_input.table_name)
        await self.psql_uploader.execute(psql_input)

        return FinanceAgentOutput(
            stocks = investment_sugggestions,
        )

In [ ]:
finance_input = FinanceAgentInput(
    # company_table_name= 'company_table',
    company_lists_csv = "https://archives.nseindia.com/content/indices/ind_nifty50list.csv",
    stock_table_name= 'stock_news_dataset',
)
agent =FinanceAgent(model=pipe)
finance_out = await agent.execute(finance_input)

2026-07-04 15:40:50,377 - INFO - Company Table loaded with 5 data points.
2026-07-04 15:40:50,377 - INFO - Company Table loaded with 5 data points.
2026-07-04 15:40:50,378 - INFO - Starting Analysis...
2026-07-04 15:40:50,378 - INFO - Starting Analysis...
Company :'Larsen & Toubro Ltd.' News article : : 0it [00:00, ?it/s]
Company :'ICICI Prudential Life Insurance Company Ltd.' News article : 100%|██████████| 48/48 [07:27<00:00,  9.33s/it]
Company :'ITI Ltd.' News article : : 0it [00:00, ?it/s]
Company :'Hyundai Motor India Ltd.' News article : : 0it [00:00, ?it/s]
Company :'HDB Financial Services Ltd.' News article : 100%|██████████| 48/48 [07:44<00:00,  9.67s/it]


In [49]:
suggestions = finance_out.stocks.suggestions

In [59]:
investment_sugggestions = pd.DataFrame(
    [[key,value['Investment'],value['Time Period'],value['Expected Profit']] for key,value in suggestions.items()],
    columns = ['Industry','Investment','Time Period','Expected Profit']
)
investment_sugggestions['Date'] = date.today()

In [60]:
investment_sugggestions

,Industry,Investment,Time Period,Expected Profit,Date
0,Larsen & Toubro Ltd.,Tentative,1-2 Years,8%,2026-07-04
1,ICICI Prudential Life Insurance Company Ltd.,Tentative,1-2 Years,12%,2026-07-04
2,ITI Ltd.,No,1-2 Years,0%,2026-07-04
3,Hyundai Motor India Ltd.,Tentative,1-2 Years,10%,2026-07-04
4,HDB Financial Services Ltd.,Tentative,1-2 Years,15%,2026-07-04


In [63]:
psql_input = UploadingInput(
    dataset = investment_sugggestions,
    uploading_columns = ['Industry','Investment',	'Time Period',	'Expected Profit',	'Date'],
    sql_column_names = ['id', 'company','investment','time_period','expected_profit','created_at'],
    table_name= 'stock_suggestions',
)

uploader = UploadToPLSQL()
# uploader._truncate_table(psql_input.table_name)
await uploader.execute(psql_input)

2026-07-04 16:39:58,582 - INFO - PSQL Connected Successfully!


2026-07-04 16:39:58,582 - INFO - PSQL Connected Successfully!
2026-07-04 16:39:58,590 - INFO - All records are uploaded to SQL!
2026-07-04 16:39:58,590 - INFO - All records are uploaded to SQL!


In [64]:
link_for_listed_company = "https://archives.nseindia.com/content/indices/ind_nifty50list.csv"
company_df = pd.read_csv(link_for_listed_company)

In [65]:
company_df

,Company Name,Industry,Symbol,Series,ISIN Code
0,Adani Enterprises Ltd.,Metals & Mining,ADANIENT,EQ,INE423A01024
1,Adani Ports and Special Economic Zone Ltd.,Services,ADANIPORTS,EQ,INE742F01042
2,Apollo Hospitals Enterprise Ltd.,Healthcare,APOLLOHOSP,EQ,INE437A01024
3,Asian Paints Ltd.,Consumer Durables,ASIANPAINT,EQ,INE021A01026
4,Axis Bank Ltd.,Financial Services,AXISBANK,EQ,INE238A01034
5,Bajaj Auto Ltd.,Automobile and Auto Components,BAJAJ-AUTO,EQ,INE917I01010
6,Bajaj Finance Ltd.,Financial Services,BAJFINANCE,EQ,INE296A01032
7,Bajaj Finserv Ltd.,Financial Services,BAJAJFINSV,EQ,INE918I01026
8,Bharat Electronics Ltd.,Capital Goods,BEL,EQ,INE263A01024
9,Bharti Airtel Ltd.,Telecommunication,BHARTIARTL,EQ,INE397D01024


In [5]:
conn = psycopg.connect(
    host=os.getenv("PS_HOST"),
    port=os.getenv("PS_PORT"),
    dbname=os.getenv("PS_DBNAME"),
    user=os.getenv("PS_USER"),
    password=os.getenv("PS_PASSWORD"),
)

In [6]:
company_table = 'company_table'
stock_table = "stock_news_dataset"
company_query = f"""
SELECT *
FROM {company_table};
"""
news_query = """
SELECT *
FROM {stock_table}
WHERE SECTOR LIKE '{industry}%'
"""
company_df= pd.read_sql_query(company_query,conn)
company_df =  company_df.sample(5)

In [7]:
market_news = dict()
for idx,row in  company_df.iterrows():
    company = row.company_name
    industry = row.industry
    market_news[company] = {
        'company' : company,
        'industry': industry,
        'score' : 0.0,
        'sentiments': [],
    }
    news_df = pd.read_sql_query(news_query.format(stock_table=stock_table,industry=industry), conn)

    for idx,row in tqdm(news_df.iterrows(),total=len(news_df), desc=f"Company :'{company}' News article : "):
        message = [
            {
                'role':'system',
                'content' : SYSTEM_PROMPT,
            },
            {
                'role':'user',
                'content': PROMPT.format(company_name = company,
                                industry = industry, 
                                title = row.title, 
                                content = row.content),
            },
        ]

        result = pipe(message)
        sentiment = json.loads(result[0]['generated_text'][-1]['content'])
        if sentiment['sentiment'].lower()!='neutral':
            market_news[company]['sentiments'].append(sentiment)
        
    market_news[company]['score'] = 0 if len(market_news[company]['sentiments'])==0 else sum([sentiment['sentiment_score'] * sentiment['confidence'] for sentiment in market_news[company]['sentiments']]) / len(market_news[company]['sentiments'])


Company :'Sundaram Finance Ltd.' News article : 100%|██████████| 48/48 [07:30<00:00,  9.38s/it]
Company :'Signatureglobal (India) Ltd.' News article : : 0it [00:00, ?it/s]
Company :'Central Depository Services (India) Ltd.' News article : 100%|██████████| 48/48 [07:41<00:00,  9.62s/it]
Company :'Premier Energies Ltd.' News article : : 0it [00:00, ?it/s]
Company :'Anupam Rasayan India Ltd.' News article : : 0it [00:00, ?it/s]


In [8]:
market_news

{'Sundaram Finance Ltd.': {'company': 'Sundaram Finance Ltd.',
  'industry': 'Financial Services',
  'score': 0.56825,
  'sentiments': [{'sentiment': 'Negative',
    'sentiment_score': 0.35,
    'confidence': 0.92,
    'event_type': 'Earnings and Revenue Performance',
    'impact': 'Medium',
    'time_horizon': 'Medium',
    'key_positive_factors': ['Strong medium-term prospects in metro and defence segments',
     'Steady revenue growth guidance of 15-25% over next three years',
     'Improving operating leverage supporting margins',
     'Rail and metro segment expected to contribute Rs 20 billion in FY27 (100% YoY growth)',
     'FY27 order inflow guidance of Rs 150 billion with closing order book of Rs 240 billion'],
    'key_negative_factors': ['Q4 FY26 net profit declined 37.4% YoY to Rs 180 crore',
     'Full-year FY26 net profit down 51.7% to Rs 141 crore',
     'EBITDA declined 35.9% YoY to Rs 271 crore',
     'EBITDA margin contracted to 15.1% from 25.6%',
     'FY26 closing 

In [9]:
#### Resoning

In [15]:
company = 'Sundaram Finance Ltd.'
resoning = market_news[company]
industry = resoning['industry']

positive_factors = [ factor for sentiment in resoning['sentiments'] for factor in sentiment['key_positive_factors']]
negagtive_factors = [ factor for sentiment in resoning['sentiments'] for factor in sentiment['key_negative_factors']]
reasonings = [
    sentiment["reasoning"]
    for sentiment in resoning["sentiments"]
    if sentiment["sentiment"].lower() != "neutral"
]


positive_factors = '  - ' +"\n  - ".join(positive_factors)
negagtive_factors = '  - ' +"\n  - ".join(negagtive_factors)
reasonings = '  - ' +"\n  - ".join(reasonings)

prompt = f"""Please analyze the news regarding the company "{company}" from "{industry}" industry.

Positive factors : 
{positive_factors}

Negagtive Factors : 
{negagtive_factors}

Reasoning : 
{reasonings}
"""

In [16]:
message  = [
    {
        'role':'system',
        'content':RESONING_SYSTEM_PROMPT,
    },
    {
        'role': 'user',
        'content': prompt,
    }
]

In [25]:
result = pipe(message)
output = json.loads(result[0]['generated_text'][-1]['content'])

In [26]:
output

{'Investment': 'Tentative',
 'Time Period': '1-2 Years',
 'Expected Profit': '12%'}

In [8]:
for idx,row in tqdm(news_df.iterrows(),total=len(news_df)):
    message = [
        {
            'role':'system',
            'content' : SYSTEM_PROMPT,
        },
        {
            'role':'user',
            'content': PROMPT.format(company_name = company,
                            industry = industry, 
                            title = row.title, 
                            content = row.content),
        },
    ]

    result = pipe(message)
    sentiment = json.loads(result[0]['generated_text'][-1]['content'])
    # if sentiment['sentiment'].lower()!='neutral':
    market_news[company]['sentiments'].append(sentiment)
    
market_news[company]['score'] = 0 if len(market_news[company]['sentiments'])==0 else sum([sentiment['sentiment_score'] * sentiment['confidence'] for sentiment in market_news[company]['sentiments']]) / len(market_news[company]['sentiments'])



100%|██████████| 17/17 [02:10<00:00,  7.68s/it]


In [12]:
market_news[company]

{'company': 'eClerx Services Ltd.',
 'industry': 'Services',
 'score': 0.0,
 'sentiments': [{'company': 'eClerx Services Ltd.',
   'sentiment': 'Neutral',
   'sentiment_score': 0.0,
   'confidence': 0.3,
   'event_type': 'Regulation',
   'impact': 'No Impact',
   'time_horizon': 'Medium',
   'key_positive_factors': [],
   'key_negative_factors': [],
   'reasoning': "The article discusses the exhaustion of the US EB-5 visa quota for Indian nationals and related regulatory changes, which is a matter of immigration policy affecting potential investors. However, eClerx Services Ltd. is a services company with no direct involvement in the EB-5 visa process, investor immigration, or related financial flows. There is no mention of eClerx's earnings, revenue, operations, or any financial or operational link to the EB-5 program. Therefore, the news has no direct or indirect financial impact on the company. The event is sector- or market-level, not company-specific. The confidence is low due to 

In [26]:


df = news_df[["title", "content"]].copy()

dataset = Dataset.from_pandas(df, preserve_index=False)

def preprocess(batch):
    return {
        "messages": [
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": PROMPT.format(
                        company_name=company,
                        industry=industry,
                        title=title,
                        content=content,
                    ),
                },
            ]
            for title, content in zip(batch["title"], batch["content"])
        ]
    }

messages = dataset.map(preprocess, batched=True)
results = pipe(messages['messages'][:], batch_size=2)

Map: 100%|██████████| 17/17 [00:00<00:00, 6804.39 examples/s]


In [25]:
print(type(messages))
print(type(messages["messages"]))

<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Column'>


In [16]:
results = pipe(
    messages["messages"],
    batch_size=2,
)

TypeError: can only concatenate str (not "Column") to str

In [10]:
non_neutral_sentiments = []
for row in tqdm(df.itertuples(index=False)):
    message = [
        {
            'role':'system',
            'content' : SYSTEM_PROMPT,
        },
        {
            'role':'user',
            'content': PROMPT.format(company_name = company,
                            industry = company, 
                            title = row.title, 
                            content = row.content),
        },
    ]

    result = pipe(message)

    sentiment = json.loads(result[0]['generated_text'][-1]['content'])

    if sentiment['sentiment'].lower()!='neutral':
        non_neutral_sentiments.append(sentiment)



NameError: name 'df' is not defined

0.675